# Loan Approval Prediction — Classificação com Métodos Clássicos

**Objetivo:** Prever se um empréstimo será aprovado (`Loan_Status`) usando KNN, Árvore de Decisão, Naive Bayes e SVM com pipelines, cross-validation e otimização de hiperparâmetros.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

## 2. Carregamento dos Dados

In [ ]:
url = "https://raw.githubusercontent.com/pabloghid/loan_approval_prediction/refs/heads/main/notebook/loan_approval.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

## 3. Análise Exploratória

In [ ]:
print("Tipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nDistribuição do target:")
print(df['Loan_Status'].value_counts())

## 4. Limpeza e Pré-processamento

### 4.1 Tratamento de valores nulos

Em vez de excluir linhas com nulos (o que reduziria o dataset), imputamos:
- **Categóricas** → moda (valor mais frequente)
- **Numéricas** → mediana (robusta a outliers)

In [ ]:
# Variáveis categóricas: substitui nulos pela moda
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Amount_Term']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Credit_History (float): substitui pela moda
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

# LoanAmount (numérica): substitui pela mediana
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

print("Nulos restantes:", df.isnull().sum().sum())

### 4.2 Conversão para numérico

Algoritmos de ML exigem entrada numérica:
- `Dependents`: '3+' → 3, depois int
- Variáveis binárias (`Gender`, `Married`, etc.) → `LabelEncoder` (0/1)
- `Property_Area` (3 categorias sem ordem) → `get_dummies` (one-hot encoding)

In [ ]:
# Remove ID (não é feature preditiva)
df = df.drop('Loan_ID', axis=1)

# Dependents: trata '3+' antes de converter para int
df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].astype(float)

# LabelEncoder para variáveis binárias/target
le = LabelEncoder()
for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Loan_Status']:
    df[col] = le.fit_transform(df[col].astype(str))

# One-hot para Property_Area (sem hierarquia entre Urban/Rural/Semiurban)
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)

print("Colunas finais:", df.columns.tolist())
df.head()

## 5. Holdout — Separação Treino/Teste

Separamos 80% para treino e 20% para teste. `stratify=y` preserva a proporção das classes em ambos os conjuntos.

In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

## 6. Modelagem com Pipelines

Cada pipeline encadeia:
1. **StandardScaler** — padroniza (média 0, desvio 1); obrigatório para KNN e SVM
2. O classificador

Avaliamos com **StratifiedKFold (5 folds)** para garantir representação balanceada das classes.

In [ ]:
pipelines = [
    ('KNN',        Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())])),
    ('DecTree',    Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(random_state=42))])),
    ('NaiveBayes', Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())])),
    ('SVM',        Pipeline([('scaler', StandardScaler()), ('clf', SVC(random_state=42))])),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipe in pipelines:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:12s} → CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

### Comparação Visual (Cross-Validation)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(cv_results.values(), labels=cv_results.keys())
ax.set_title('Comparação de Modelos — Cross-Validation (5 folds)')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Otimização de Hiperparâmetros (GridSearchCV)

Testamos combinações dos principais hiperparâmetros de cada algoritmo usando o mesmo `cv` de 5 folds.

In [ ]:
param_grids = {
    'KNN': {
        'clf__n_neighbors': [3, 5, 7, 11],
        'clf__weights':     ['uniform', 'distance'],
    },
    'DecTree': {
        'clf__max_depth':        [3, 5, 7, None],
        'clf__min_samples_leaf': [1, 5, 10],
    },
    'NaiveBayes': {
        'clf__var_smoothing': [1e-9, 1e-7, 1e-5],
    },
    'SVM': {
        'clf__C':      [0.1, 1, 10],
        'clf__kernel': ['linear', 'rbf'],
    },
}

best_models = {}

for name, pipe in pipelines:
    grid = GridSearchCV(pipe, param_grids[name], cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    print(f"{name:12s} → Best CV: {grid.best_score_:.4f} | {grid.best_params_}")

## 8. Avaliação Final no Conjunto de Teste (Holdout)

In [ ]:
print(f"{'Modelo':<12} {'Accuracy':>10}")
print("-" * 24)

test_scores = {}
for name, model in best_models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    test_scores[name] = acc
    print(f"{name:<12} {acc:>10.4f}")

In [ ]:
# Relatório detalhado do melhor modelo
best_name = max(test_scores, key=test_scores.get)
best_model = best_models[best_name]

print(f"Melhor modelo: {best_name} (Accuracy = {test_scores[best_name]:.4f})\n")
print(classification_report(
    y_test, best_model.predict(X_test),
    target_names=['Reprovado', 'Aprovado']
))

## 9. Exportação do Melhor Modelo

In [ ]:
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f"Modelo '{best_name}' exportado como 'best_model.pkl'")

## 10. Predição no test.csv

O `test.csv` não possui `Loan_Status` — é o conjunto de inferência real. Aplicamos as mesmas transformações do treino e geramos as predições.

In [ ]:
df_test = pd.read_csv('test.csv')
loan_ids = df_test['Loan_ID'].copy()

# 1. Imputação (mesma lógica do treino)
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Amount_Term']:
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])
df_test['Credit_History'] = df_test['Credit_History'].fillna(df_test['Credit_History'].mode()[0])
df_test['LoanAmount'] = df_test['LoanAmount'].fillna(df_test['LoanAmount'].median())

# 2. Remover ID e converter tipos
df_test = df_test.drop('Loan_ID', axis=1)
df_test['Dependents'] = df_test['Dependents'].replace('3+', '3').astype(int)
df_test['Loan_Amount_Term'] = df_test['Loan_Amount_Term'].astype(float)

# 3. LabelEncoder (reutiliza mesmo `le` do treino)
for col in ['Gender', 'Married', 'Education', 'Self_Employed']:
    df_test[col] = le.fit_transform(df_test[col].astype(str))

# 4. One-hot Property_Area
df_test = pd.get_dummies(df_test, columns=['Property_Area'], drop_first=True)

# Garante mesmas colunas do treino (caso alguma categoria não apareça no test)
for col in X.columns:
    if col not in df_test.columns:
        df_test[col] = 0
df_test = df_test[X.columns]

# 5. Predição
predictions = best_model.predict(df_test)
pred_labels = ['Y' if p == 1 else 'N' for p in predictions]

result = pd.DataFrame({'Loan_ID': loan_ids, 'Loan_Status': pred_labels})
result.to_csv('predictions.csv', index=False)

print(f"Predições geradas: {len(result)} amostras")
print(result['Loan_Status'].value_counts())
result.head(10)